# Classification — Early Settlement: SUN vs SOL

---

## Project Overview

**Objective:** Binary classification to distinguish between two types of early settlement:
- `SAN (1)` — client settled the contract early on their own initiative
- `SOL (0)` — client settled early due to refinancing or other external reason

**Data:** Aggregated dataset — 1 row per client, merged from 4 source tables:
`BDOSSTOTAL` · `CRC` · `CredScore` · `FAMA_LIGHT_`

---

## Notebook Structure

```
┌─────────────────────────────────────────────────────────────┐
│  0. SET UP                                                  │
│     Imports, constants, paths                               │
├─────────────────────────────────────────────────────────────┤
│  1. LOAD DATASET                                            │
│     Load aggregated client dataset (1 row per client)       │
├─────────────────────────────────────────────────────────────┤
│  2. TRAIN / TEST SPLIT                                      │
│     Split once — X_test goes in the drawer until Section 8  │
│     Define CV strategy (StratifiedKFold, k=5)               │
├─────────────────────────────────────────────────────────────┤
│  3. PREPROCESSING PIPELINE                                  │
│     ClientDataCleaner                                       │
│       → ClientOutlierHandler                                │
│         → ClientImputer                                     │
│           → ClientFeatureEngineer                           │
│             → ColumnTransformer  (encoding + scaling)[TODO] │
├─────────────────────────────────────────────────────────────┤
│  4. FIT & TRANSFORM                                         │
│     fit_transform(X_train)  ← learns from train only        │
│     transform(X_test)       ← applies, never learns         │
├─────────────────────────────────────────────────────────────┤
│  5. FEATURE SELECTION                  [TODO]               │
│     Computed on X_train_df only                             │
│     Same mask applied to X_test_df                          │
├─────────────────────────────────────────────────────────────┤
│  6. MODEL COMPARISON                   [TODO]               │
│     cross_validate on X_train_fs                            │
│     Compare baseline: LR · RF · GBM                         │
├─────────────────────────────────────────────────────────────┤
│  7. HYPERPARAMETER TUNING              [TODO]               │
│     RandomizedSearchCV on best model                        │
│     Still only uses X_train_fs                              │
├─────────────────────────────────────────────────────────────┤
│  8. FINAL EVALUATION  ← runs ONCE at the very end           │
│     predict(X_test_fs)                                      │
│     ROC-AUC · Classification Report · Confusion Matrix      │
└─────────────────────────────────────────────────────────────┘
```

---

## Anti-Leakage Rules

| Rule | Where enforced |
|---|---|
| `X_test` never influences any decision | Split in Section 2, only used in Section 8 |
| All statistics learned from train fold only | `fit_transform` only on `X_train` in Section 4 |
| Feature selection criteria computed on train only | Section 5 uses `X_train_df` exclusively |
| CV uses only `X_train_fs` | Sections 6 and 7 never touch `X_test_fs` |


## 0. SET UP

In [ ]:
import io
from pydoc import cli

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay, roc_auc_score, classification_report

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline

from src.code.class_pipeline_functions import ClientDataCleaner, ClientOutlierHandler, ClientImputer, \
    ClientFeatureEngineer

RANDOM_STATE = 42
pd.set_option('display.max_columns', None)

CLIENT_PATH = io.data_path("")

## 1. Load Aggregation Dataset

In [ ]:
client_data = io.load(CLIENT_PATH)
client_data.head(3)

## 2. Train/Test Split

In [ ]:
X = client_data ##drop target
y = client_data['target']


X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,                  # maintain the portion of SAN/SOL
    random_state=RANDOM_STATE,
)

print(f'\nTrain : {X_train.shape} — SAN rate: {y_train.mean():.2%}')
print(f'Test  : {X_test.shape}  — SAN rate: {y_test.mean():.2%}')

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

NameError: name 'client_data' is not defined

## 3. Build Preprocessing Pipeline

No model here, this pipeline only preprocesses the data.
The model is added later after feature selection.

In [ ]:
preprocessing_pipe = Pipeline([
    ('cleaning',  ClientDataCleaner( ##probably not necessary
                     datetime_cols  = ['DPOS', 'DCREAT', 'DATFIN', 'D1FIN'],
                     cols_to_remove = [],
                     verbose        = True,
                 )),
    ('outliers', ClientOutlierHandler(
                     methods   = ('iqr', 'mod_z'),
                     min_votes = 2,
                     iqr_k     = 1.5,
                     verbose   = True,
                 )),
    ('imputer',  ClientImputer(
                     numeric_strategy  = 'median',
                     datetime_strategy = 'ffill', ## not sure if this makes sense, but we don't want to drop date cols before feature engineering
                     verbose           = True,
                 )),
    ('fe',       ClientFeatureEngineer(
                     include_composite = True,
                     drop_date_cols    = True,
                     verbose           = True,
                 )),
    ## TODO ('ct',       ct),
])

print('Pipeline built successfully')
print(preprocessing_pipe)

## 4. Fit & Transform

- `fit_transform` on `X_train`: pipeline learns all statistics from training data only
- `transform` on `X_test`: applies without learning anything new

**Never call `fit_transform` on `X_test`.**

In [ ]:
# Fit on train, transform both
X_train_processed = preprocessing_pipe.fit_transform(X_train, y_train)
X_test_processed  = preprocessing_pipe.transform(X_test)

# Get feature names from the ColumnTransformer
feature_names = preprocessing_pipe.named_steps['ct'].get_feature_names_out()

# Convert to DataFrame for easier inspection and feature selection
X_train_df = pd.DataFrame(X_train_processed, columns=feature_names)
X_test_df  = pd.DataFrame(X_test_processed,  columns=feature_names)

print(f'X_train processed : {X_train_df.shape}')
print(f'X_test  processed : {X_test_df.shape}')
print(f'Any NaNs remaining: {X_train_df.isna().sum().sum()}') ##just to check

## 5. Feature Selection

Feature selection is applied on `X_train_df` only.
The same selected columns are then applied to `X_test_df`.

**No leakage here because:**
- All selection criteria (variance, correlation, importance) are computed on `X_train_df`
- `X_test_df` is only filtered at the end using the mask learned from train

In [ ]:
## TODO


# Placeholder — keeps all features until you decide
selected_features = feature_names.tolist()

X_train_fs = X_train_df[selected_features]
X_test_fs  = X_test_df[selected_features]

print(f'Features after selection: {len(selected_features)}')

## 6. Model Comparison

Quick baseline comparison across models using cross-validation on `X_train_fs`.
Pick the best baseline before investing time in hyperparameter tuning.


## 7. Hyperparameter Tuning

Run `RandomizedSearchCV` on the best model from Section 6.
Still using only `X_train_fs`. `X_test_fs` remains untouched.

## 8. Final Evaluation on Test Set

**This section runs only once at the very end.**
The test set was never used for any decision up to this point.

In [ ]:
final_model = search.best_estimator_ ##it comes from the randomized search

y_pred      = final_model.predict(X_test_fs)
y_pred_prob = final_model.predict_proba(X_test_fs)[:, 1]

print('=' * 50)
print('FINAL TEST SET RESULTS')
print('=' * 50)
print(f'ROC-AUC  : {roc_auc_score(y_test, y_pred_prob):.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['SOL', 'SUN']))

# Confusion matrix
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['SOL', 'SUN'],
    ax=ax,
)
plt.title('Confusion Matrix — Test Set')
plt.tight_layout()
plt.show()